# Discesa stocastica del gradiente

> **Nota AI Lab — ambiente di lettura.** Il testo seguente è la traduzione del notebook Kaggle originale conservato nel repository. I riferimenti a pulsanti, celle nascoste e invii descrivono l’ambiente Kaggle dell’originale; qui le celle di codice sono visibili. Per eseguire i notebook fuori da Kaggle, segui la guida AI Lab all’ambiente.



## Preparazione in Colab

1. Salvare una copia del notebook nel proprio Drive.
2. Eseguire la cella seguente e attendere «Preparazione completata».
3. Eseguire le celle nell’ordine e svolgere gli esercizi.

La cella di preparazione acquisisce i dataset e i controlli degli esercizi dal repository del corso e configura l’ambiente Python. Dopo il riavvio della sessione occorre eseguire nuovamente la preparazione.

In [ ]:
from pathlib import Path
import hashlib
import importlib.util
from urllib.request import urlopen
from IPython.display import display, Markdown

course_source = 'https://raw.githubusercontent.com/prof-davidaffo/kaggle-learntools/master'
course_root = (Path("/content") if Path("/content").is_dir() else Path.home()) / "ai-lab-colab"
course_root.mkdir(parents=True, exist_ok=True)
bootstrap_path = course_root / "_bootstrap.py"
bootstrap_hash = '56f1300be5bf273a70dd4a30e704b20f642ec01dacc4d2d4b5a295a02e1da60f'
if not bootstrap_path.is_file() or hashlib.sha256(bootstrap_path.read_bytes()).hexdigest() != bootstrap_hash:
    with urlopen(course_source + "/ai_lab/tools/colab_runtime.py", timeout=90) as response:
        bootstrap_data = response.read()
    if hashlib.sha256(bootstrap_data).hexdigest() != bootstrap_hash:
        raise RuntimeError("Course setup changed. Reopen the latest notebook from the course site.")
    bootstrap_path.write_bytes(bootstrap_data)
bootstrap_spec = importlib.util.spec_from_file_location("ai_lab_colab_bootstrap", bootstrap_path)
bootstrap = importlib.util.module_from_spec(bootstrap_spec)
bootstrap_spec.loader.exec_module(bootstrap)
bootstrap.download_runtime(course_source + "/ai_lab/colab/runtime.json", course_source, course_root)
bootstrap.setup(course_root, globals(), deep_learning=True)
display(Markdown("**Preparazione completata.** Puoi eseguire le celle della lezione."))


# Introduzione

In questo esercizio addestrerai una rete neurale sul dataset *Fuel Economy* e poi esplorerai l’effetto del tasso di apprendimento e della dimensione del batch su SGD.

Quando sei pronto, esegui la prossima cella per preparare tutto!


In [ ]:
# Configura i grafici.
import matplotlib.pyplot as plt
from learntools.deep_learning_intro.dltools import animate_sgd
plt.style.use('seaborn-v0_8-whitegrid')
# Imposta i valori predefiniti di Matplotlib.
plt.rc('figure', autolayout=True)
plt.rc('axes', labelweight='bold', labelsize='large',
       titleweight='bold', titlesize=18, titlepad=10)
plt.rc('animation', html='html5')

# Configura il sistema di verifica degli esercizi.
from learntools.core import binder
binder.bind(globals())
from learntools.deep_learning_intro.ex3 import *

Nel dataset *Fuel Economy*, il compito è prevedere l’efficienza nei consumi di un’automobile a partire da feature come il tipo di motore o l’anno di produzione.

Per prima cosa, carica il dataset eseguendo la cella seguente.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import make_column_transformer, make_column_selector
from sklearn.model_selection import train_test_split

fuel = pd.read_csv('../input/dl-course-data/fuel.csv')

X = fuel.copy()
# Rimuovi il target.
y = X.pop('FE')

preprocessor = make_column_transformer(
    (StandardScaler(),
     make_column_selector(dtype_include=np.number)),
    (OneHotEncoder(sparse=False),
     make_column_selector(dtype_include=object)),
)

X = preprocessor.fit_transform(X)
y = np.log(y) # Applica il logaritmo al target invece della standardizzazione.

input_shape = [X.shape[1]]
print("Input shape: {}".format(input_shape))

Se vuoi, dai un’occhiata ai dati. In questo caso il target è la colonna `'FE'` e le colonne rimanenti sono le feature.


In [ ]:
# Rimuovi il simbolo di commento per vedere i dati originali.
fuel.head()
# Rimuovi il simbolo di commento per vedere le feature elaborate.
pd.DataFrame(X[:10,:]).head()

Esegui la prossima cella per definire la rete che useremo per questo compito.


In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=input_shape),
    layers.Dense(128, activation='relu'),    
    layers.Dense(64, activation='relu'),
    layers.Dense(1),
])

# 1) Aggiungere perdita e ottimizzatore

Prima di addestrare la rete dobbiamo definire la perdita e l’ottimizzatore. Con il metodo `compile` del modello, aggiungi l’ottimizzatore Adam e la perdita MAE.


In [ ]:
# SCRIVI QUI IL CODICE
____

# Controlla la risposta.
q_1.check()

In [ ]:
# Le righe seguenti mostrano un suggerimento o il codice della soluzione.
#q_1.hint()
#q_1.solution()

# 2) Addestrare il modello

Dopo aver definito il modello e averlo compilato con perdita e ottimizzatore, sei pronto per il training. Addestra la rete per 200 epoche con batch di dimensione 128. I dati in ingresso sono `X`, con target `y`.


In [ ]:
# SCRIVI QUI IL CODICE
history = ____

# Controlla la risposta.
q_2.check()

In [ ]:
# Le righe seguenti mostrano un suggerimento o il codice della soluzione.
#q_2.hint()
#q_2.solution()

L’ultimo passo è osservare le curve di perdita e valutare l’addestramento. Esegui la cella seguente per ottenere il grafico della perdita di training.


In [ ]:
import pandas as pd

history_df = pd.DataFrame(history.history)
# Inizia il grafico dall'epoca 5; puoi cambiare questo valore per modificare la visualizzazione.
history_df.loc[5:, ['loss']].plot();

# 3) Valutare l’addestramento

Se addestrassi il modello più a lungo, ti aspetteresti che la perdita diminuisse ancora?


In [ ]:
# Mostra la soluzione; esegui questa cella per convalidare l'esercizio.
q_3.check()

Con il tasso di apprendimento e la dimensione del batch puoi controllare, in parte:

- Quanto tempo serve per addestrare un modello.
- Quanto sono rumorose le curve di apprendimento.
- Quanto diventa piccola la perdita.

Per comprendere meglio questi due parametri osserveremo il modello lineare, la nostra rete neurale più semplice. Avendo un solo peso e un bias, permette di vedere più facilmente gli effetti di una modifica dei parametri.

La prossima cella genererà un’animazione simile a quella della lezione. Modifica `learning_rate`, `batch_size` e `num_examples`, il numero di punti dei dati, poi eseguila: potrebbe richiedere qualche istante. Prova le combinazioni seguenti oppure inventane altre:

| `learning_rate` | `batch_size` | `num_examples` |
| --- | --- | --- |
| 0.05 | 32 | 256 |
| 0.05 | 2 | 256 |
| 0.05 | 128 | 256 |
| 0.02 | 32 | 256 |
| 0.2 | 32 | 256 |
| 1.0 | 32 | 256 |
| 0.9 | 4096 | 8192 |
| 0.99 | 4096 | 8192 |


In [ ]:
# SCRIVI QUI IL CODICE: prova valori diversi di learning rate, batch size e numero di esempi.
learning_rate = 0.05
batch_size = 32
num_examples = 256

animate_sgd(
    learning_rate=learning_rate,
    batch_size=batch_size,
    num_examples=num_examples,
    # Puoi modificare anche questi valori.
    steps=50, # Numero totale di passi di training, cioè batch elaborati.
    true_w=3.0, # Pendenza dei dati.
    true_b=2.0, # Termine costante dei dati.
)

# 4) Tasso di apprendimento e dimensione del batch

Qual è stato l’effetto delle modifiche a questi parametri? Dopo averci riflettuto, esegui la cella seguente per leggere una discussione.


In [ ]:
# Mostra la soluzione; esegui questa cella per convalidare l'esercizio.
q_4.check()

# Prosegui

Impara a [**migliorare le prestazioni del modello**](https://prof-davidaffo.github.io/kaggle-learntools/corsi/deep_learning_intro/tut4.html) regolandone la capacità o aggiungendo una callback di early stopping.
